# A Finite Calculation from Supplied Priors

## Discretizing monthly return laws

For month \(j\), let the supplied normal return law have mean \(\mu_j\) and standard deviation \(\sigma_j>0\). If the interior bin has lower edge \(a_k\) and upper edge \(b_k\), its probability is

$$
p_{j,k}=\Phi\left(\frac{b_k-\mu_j}{\sigma_j}\right)-\Phi\left(\frac{a_k-\mu_j}{\sigma_j}\right).
$$

Here, \(p_{j,k}\) is the probability assigned by monthly law \(j\) to bin \(k\); \(\Phi\) is the standard normal cumulative distribution function; \(a_k\) and \(b_k\) are bin edges; \(\mu_j\) is the monthly mean daily return; and \(\sigma_j\) is the monthly population standard deviation. The lower and upper tail probabilities are included as separate bins, so every row sums to one.

In [1]:
from pathlib import Path
import calendar
import hashlib
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)
from scipy.stats import norm


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset

import RiskLabAI
from RiskLabAI.features.entropy_features.ambiguity import normal_prior_bin_probabilities, probability_ambiguity

assert RiskLabAI.__version__ == "3.2.0"

DATA_PATH = ensure_dataset(BOOK_ROOT, "fred_nasdaq")
FIGURES = BOOK_ROOT / "outputs" / "figures" / "04_probability_ambiguity"
TABLES = BOOK_ROOT / "outputs" / "tables" / "04_probability_ambiguity"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "04_probability_ambiguity"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return "0"
        if number.is_integer():
            return f"{int(number):,}"
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f"{number:.6g}"
        return f"{number:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    shown = frame.apply(lambda column: column.map(rendered_value))
    for column in frame.columns:
        if "year" in str(column).lower():
            shown[column] = frame[column].map(lambda value: "" if pd.isna(value) else str(int(value)) if isinstance(value, (int, float, np.integer, np.floating)) and float(value).is_integer() else str(value))
    return shown


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)


def load_public_data():
    expected_hash = "2E988948FFCB108DA4902DFEDAA5DF756E7E102DC523A99A8916A0533291F2D7"
    assert DATA_PATH.is_file()
    assert sha256(DATA_PATH) == expected_hash
    raw = pd.read_csv(DATA_PATH)
    raw["Date"] = pd.to_datetime(raw["observation_date"])
    raw["NASDAQ Composite"] = pd.to_numeric(raw["NASDAQCOM"], errors="coerce")
    data = raw.dropna(subset=["NASDAQ Composite"]).sort_values("Date").reset_index(drop=True)
    data["Daily log return"] = np.log(data["NASDAQ Composite"] / data["NASDAQ Composite"].shift(1))
    data["Year"] = data["Date"].dt.year
    data["Month"] = data["Date"].dt.month
    return data


def monthly_moments(data):
    returns = data.dropna(subset=["Daily log return"]).copy()
    moments = returns.groupby(["Year", "Month"])["Daily log return"].agg(
        Mean="mean",
        Standard_deviation=lambda values: values.std(ddof=0),
        Observations="size",
    ).reset_index()
    return moments


def normal_edges(width):
    intervals = int(round(0.12 / width))
    return np.linspace(-0.06, 0.06, intervals + 1)


def bin_labels(edges):
    labels = [f"Below {100 * edges[0]:g}%"]
    labels.extend(f"{100 * left:g}% to {100 * right:g}%" for left, right in zip(edges[:-1], edges[1:]))
    labels.append(f"Above {100 * edges[-1]:g}%")
    return labels

In [2]:
data = load_public_data()
moments = monthly_moments(data)
worked = moments.loc[moments["Year"] == 1994].copy().reset_index(drop=True)
assert len(worked) == 12
worked["Month name"] = worked["Month"].map(lambda value: calendar.month_abbr[int(value)])

monthly_table = worked[["Month name", "Observations"]].copy()
monthly_table["Mean daily return (%)"] = 100 * worked["Mean"]
monthly_table["Daily standard deviation (%)"] = 100 * worked["Standard_deviation"]
save_table(monthly_table, "monthly_priors_1994", "Monthly normal laws estimated from public daily returns in 1994.", "tab:chapter5-monthly-priors-1994")

selection = pd.DataFrame({
    "Selection rule": ["First complete calendar year by date"],
    "Year": [1994],
    "Monthly laws": [len(worked)],
    "Daily returns": [worked["Observations"].sum()],
    "Minimum returns in a month": [worked["Observations"].min()],
    "Maximum returns in a month": [worked["Observations"].max()],
})
save_table(selection, "worked_year_selection", "Date based selection of the worked decomposition year.", "tab:chapter5-worked-year-selection", fit=True)

edges = normal_edges(0.002)
labels = bin_labels(edges)
probabilities = normal_prior_bin_probabilities(worked["Mean"], worked["Standard_deviation"], edges=edges)
prior_weights = np.full(len(worked), 1 / len(worked))
result = probability_ambiguity(probabilities, prior_weights=prior_weights, bin_width=0.002)
assert probabilities.shape == (12, 62)
assert np.all(probabilities >= 0)
assert np.allclose(probabilities.sum(axis=1), 1)

x_grid = np.linspace(-0.06, 0.06, 601)
density_rows = []
for month, mean, standard_deviation in zip(worked["Month name"], worked["Mean"], worked["Standard_deviation"]):
    density = norm.pdf(x_grid, loc=mean, scale=standard_deviation)
    for value, density_value in zip(x_grid, density):
        density_rows.append({"Month": month, "Daily return (%)": 100 * value, "Density": density_value / 100})
density_frame = pd.DataFrame(density_rows)
density_frame.to_csv(FIGURE_DATA / "monthly_normal_densities_1994.csv", index=False)

matrix_rows = []
for month_index, month in enumerate(worked["Month name"]):
    for bin_index, (label, probability) in enumerate(zip(labels, probabilities[month_index])):
        matrix_rows.append({"Month": month, "Bin number": bin_index + 1, "Bin": label, "Probability": probability})
matrix_frame = pd.DataFrame(matrix_rows)
matrix_frame.to_csv(FIGURE_DATA / "bin_probability_matrix_1994.csv", index=False)

,Month name,Observations,Mean daily return (%),Daily standard deviation (%)
0,Jan,20,0.18911,0.342269
1,Feb,19,-0.052666,0.788022
2,Mar,23,-0.277728,0.752982
3,Apr,19,-0.068547,1.228222
4,May,21,0.008752,0.675294
5,Jun,22,-0.184411,0.779227
6,Jul,20,0.113441,0.557226
7,Aug,23,0.254084,0.382399
8,Sep,21,-0.008279,0.621371
9,Oct,21,0.08154,0.67394


,Selection rule,Year,Monthly laws,Daily returns,Minimum returns in a month,Maximum returns in a month
0,First complete calendar year by date,1994,12,251,19,23


In [3]:
figure, axis = plt.subplots(figsize=(10.8, 6.2), constrained_layout=True)
month_colors = plt.cm.viridis(np.linspace(0.05, 0.95, 12))
for month, color in zip(worked["Month name"], month_colors):
    part = density_frame[density_frame["Month"] == month]
    axis.plot(part["Daily return (%)"], part["Density"], color=color, linewidth=1.35, label=month)
axis.set_xlabel("Daily log return (%)")
axis.set_ylabel("Density")
axis.set_title("Twelve Monthly Normal Laws in 1994")
axis.legend(ncol=4, fontsize=8.5)
finish_axis(axis)
save_figure(figure, "monthly_normal_laws_1994")
display(figure)
plt.close(figure)

<Figure size 1080x620 with 1 Axes>

In [4]:
figure, axis = plt.subplots(figsize=(11.0, 5.7), constrained_layout=True)
image = axis.imshow(probabilities, aspect="auto", cmap="magma", interpolation="nearest")
axis.set_yticks(np.arange(12), worked["Month name"])
tick_positions = [0, 11, 21, 31, 41, 51, 61]
axis.set_xticks(tick_positions, [labels[index] for index in tick_positions], rotation=35, ha="right")
axis.set_xlabel("Return bin")
axis.set_ylabel("Monthly law")
axis.set_title("Probability Assigned to Each Exhaustive Bin")
colorbar = figure.colorbar(image, ax=axis)
colorbar.set_label("Probability")
save_figure(figure, "bin_probability_matrix_1994")
display(figure)
plt.close(figure)

<Figure size 1100x570 with 2 Axes>

## Decomposing ambiguity across states

With equal weights \(q_j=1/12\), the finite measure is built in three steps:

$$
\bar p_k=\sum_{j=1}^{12}q_jp_{j,k},
\qquad
v_k=\sum_{j=1}^{12}q_j(p_{j,k}-\bar p_k)^2,
\qquad
c_k=\bar p_kv_k.
$$

Here, \(\bar p_k\) is the average probability of bin \(k\); \(v_k\) is the population variance of that probability across the twelve laws; and \(c_k\) is the contribution of the bin. The unscaled ambiguity measure is the sum \(A=\sum_kc_k\). A bin contributes only when it has both positive average probability and disagreement across laws.

In [5]:
mean_probability = result["mean_probabilities"]
probability_variance = result["probability_variances"]
contribution = mean_probability * probability_variance
contribution_frame = pd.DataFrame({
    "Bin number": np.arange(1, len(labels) + 1),
    "Bin": labels,
    "Mean probability": mean_probability,
    "Probability variance": probability_variance,
    "Contribution": contribution,
})
contribution_frame.to_csv(FIGURE_DATA / "bin_contributions_1994.csv", index=False)

top_contributions = contribution_frame.nlargest(10, "Contribution")[["Bin", "Mean probability", "Probability variance", "Contribution"]]
save_table(top_contributions, "largest_bin_contributions_1994", "Ten largest bin contributions to the 1994 ambiguity measure.", "tab:chapter5-largest-bin-contributions-1994", fit=True)

selected_indices = [0, 21, 26, 30, 31, 35, 40, 61]
excerpt = pd.DataFrame({"Bin": [labels[index] for index in selected_indices]})
for month_index in [0, 3, 7, 11]:
    excerpt[worked.loc[month_index, "Month name"]] = probabilities[month_index, selected_indices]
save_table(excerpt, "bin_probability_excerpt_1994", "Selected bin probabilities for four monthly laws.", "tab:chapter5-bin-probability-excerpt-1994", fit=True)

summary = pd.DataFrame({
    "Quantity": ["Monthly laws", "Interior bins", "Tail bins", "Unscaled ambiguity", "Scaled ambiguity", "Interior bin width"],
    "Value": [12, 60, 2, result["unscaled"], result["scaled"], 0.002],
})
save_table(summary, "ambiguity_summary_1994", "Summary of the 1994 finite ambiguity calculation.", "tab:chapter5-ambiguity-summary-1994")

,Bin,Mean probability,Probability variance,Contribution
32,0.2% to 0.4%,0.116462,0.002135,0.000249
31,0% to 0.2%,0.123419,0.001758,0.000217
33,0.4% to 0.6%,0.095135,0.001095,0.000104
30,-0.2% to 0%,0.114807,0.000523,5.99951e-05
28,-0.6% to -0.4%,0.07794,0.000546,4.25794e-05
27,-0.8% to -0.6%,0.060137,0.000644,3.87535e-05
26,-1% to -0.8%,0.044561,0.000506,2.25631e-05
29,-0.4% to -0.2%,0.097256,0.000225,2.1896e-05
34,0.6% to 0.8%,0.068325,0.000253,1.72851e-05
25,-1.2% to -1%,0.031497,0.000331,1.04388e-05


,Bin,Jan,Apr,Aug,Dec
0,Below -6%,2.18583e-73,6.85067e-07,2.00745e-60,5.63762e-16
1,-2% to -1.8%,3.01529e-09,0.0214,3.71474e-08,0.004222
2,-1% to -0.8%,0.001671,0.051629,0.002401,0.051
3,-0.2% to 0%,0.162497,0.06487,0.135679,0.104912
4,0% to 0.2%,0.222394,0.064284,0.190562,0.105298
5,0.8% to 1%,0.028231,0.047583,0.051151,0.052714
6,1.8% to 2%,1.19916e-06,0.018013,2.39323e-05,0.004527
7,Above 6%,6.02144e-65,3.88773e-07,2.48084e-51,7.06061e-16


,Quantity,Value
0,Monthly laws,12
1,Interior bins,60
2,Tail bins,2
3,Unscaled ambiguity,0.000798
4,Scaled ambiguity,0.399782
5,Interior bin width,0.002


,Quantity,Value
0,Monthly laws,12
1,Interior bins,60
2,Tail bins,2
3,Unscaled ambiguity,0.000798
4,Scaled ambiguity,0.399782
5,Interior bin width,0.002


In [6]:
figure, axes = plt.subplots(3, 1, figsize=(11.0, 8.3), sharex=True, constrained_layout=True)
positions = contribution_frame["Bin number"]
axes[0].plot(positions, mean_probability, color=PALETTE["cyan"], linewidth=1.8)
axes[0].set_ylabel("Mean probability")
axes[0].set_title("Average Probability, Probability Variance, and Bin Contribution")
axes[1].plot(positions, probability_variance, color=PALETTE["amber"], linewidth=1.8)
axes[1].set_ylabel("Probability variance")
axes[2].bar(positions, contribution, color=PALETTE["green"], width=0.85)
axes[2].set_ylabel("Contribution")
axes[2].set_xlabel("Exhaustive bin number")
for axis in axes:
    finish_axis(axis)
save_figure(figure, "bin_contributions_1994")
display(figure)
plt.close(figure)

<Figure size 1100x830 with 3 Axes>

## Validating the ambiguity calculation

The contribution of disagreement between laws \(i\) and \(j\) can be written as

$$
C_{ij}=\frac{q_iq_j}{2}\sum_{k=1}^{K}\bar p_k(p_{i,k}-p_{j,k})^2.
$$

Here, \(C_{ij}\) is the ordered pair contribution; \(q_i\) and \(q_j\) are the two prior weights; and the remaining symbols retain their earlier meanings. Summing \(C_{ij}\) over every ordered pair gives \(A\). The diagonal is zero because a law cannot disagree with itself.

For interior bin width \(h\), the reported scaled measure is

$$
A^{\mathrm{scaled}}=\frac{A}{h(1-h)}.
$$

Here, \(A^{\mathrm{scaled}}\) is the scaled value and \(0<h<1\) is the interior bin width expressed in return units. Scaling reduces the mechanical size effect of the grid, but it does not make the value independent of the grid.

In [7]:
pair_matrix = np.zeros((12, 12))
for first in range(12):
    for second in range(12):
        pair_matrix[first, second] = 0.5 * prior_weights[first] * prior_weights[second] * np.sum(
            mean_probability * (probabilities[first] - probabilities[second]) ** 2
        )
assert np.isclose(pair_matrix.sum(), result["unscaled"])

pair_rows = []
for first in range(12):
    for second in range(first + 1, 12):
        pair_rows.append({
            "First month": worked.loc[first, "Month name"],
            "Second month": worked.loc[second, "Month name"],
            "Combined pair contribution": 2 * pair_matrix[first, second],
        })
pair_frame = pd.DataFrame(pair_rows)
save_table(pair_frame.nlargest(10, "Combined pair contribution"), "largest_month_pairs_1994", "Ten monthly pairs with the largest combined contribution.", "tab:chapter5-largest-month-pairs-1994")

pair_data = pd.DataFrame(pair_matrix, index=worked["Month name"], columns=worked["Month name"])
pair_data.index.name = "First month"
pair_data.to_csv(FIGURE_DATA / "pairwise_dispersion_1994.csv")

scaled_probabilities = normal_prior_bin_probabilities(100 * worked["Mean"], 100 * worked["Standard_deviation"], edges=100 * edges)
unit_difference = float(np.max(np.abs(probabilities - scaled_probabilities)))
permutation = np.array([11, 0, 7, 3, 9, 2, 6, 1, 10, 5, 8, 4])
permuted = probability_ambiguity(probabilities[permutation], prior_weights=prior_weights[permutation], bin_width=0.002)
direct_value = float(mean_probability @ probability_variance)
checks = pd.DataFrame({
    "Check": [
        "Every monthly probability row sums to one",
        "All bin probabilities are nonnegative",
        "Direct sum agrees with the library value",
        "Pairwise identity agrees with the library value",
        "Permutation leaves the value unchanged",
        "Percent and decimal units give the same probabilities",
        "Scaled value follows the stated relation",
    ],
    "Maximum difference": [
        np.max(np.abs(probabilities.sum(axis=1) - 1)),
        max(0, -probabilities.min()),
        abs(direct_value - result["unscaled"]),
        abs(pair_matrix.sum() - result["unscaled"]),
        abs(permuted["unscaled"] - result["unscaled"]),
        unit_difference,
        abs(result["scaled"] - result["unscaled"] / (0.002 * 0.998)),
    ],
    "Result": ["Pass"] * 7,
})
assert np.all(checks["Maximum difference"] < 1e-12)
save_table(checks, "finite_identity_checks_1994", "Checks of the 1994 finite calculation.", "tab:chapter5-finite-identity-checks-1994", fit=True)

grid = pd.DataFrame({
    "Item": ["Lower finite edge", "Upper finite edge", "Interior bin width", "Interior bins", "Tail bins", "Total bins"],
    "Value": [-0.06, 0.06, 0.002, 60, 2, 62],
})
save_table(grid, "default_grid_definition", "Default exhaustive grid for the finite calculation.", "tab:chapter5-default-grid-definition")

units = pd.DataFrame({
    "Input representation": ["Decimal returns", "Percentage point returns"],
    "Mean multiplier": [1, 100],
    "Standard deviation multiplier": [1, 100],
    "Edge multiplier": [1, 100],
    "Maximum probability difference": [0, unit_difference],
})
save_table(units, "return_unit_consistency", "Equivalent normal-bin probabilities under consistent return units.", "tab:chapter5-return-unit-consistency", fit=True)

,First month,Second month,Combined pair contribution
2,Jan,Apr,5.90421e-05
1,Jan,Mar,4.74343e-05
33,Apr,Aug,4.68786e-05
4,Jan,Jun,4.31058e-05
25,Mar,Aug,3.99659e-05
9,Jan,Nov,3.84911e-05
0,Jan,Feb,3.8084e-05
46,Jun,Aug,3.52598e-05
10,Jan,Dec,3.37383e-05
58,Aug,Nov,3.23799e-05


,Check,Maximum difference,Result
0,Every monthly probability row sums to one,1.11022e-16,Pass
1,All bin probabilities are nonnegative,0,Pass
2,Direct sum agrees with the library value,0,Pass
3,Pairwise identity agrees with the library value,2.1684e-19,Pass
4,Permutation leaves the value unchanged,0,Pass
5,Percent and decimal units give the same probab...,1.11022e-16,Pass
6,Scaled value follows the stated relation,0,Pass


,Item,Value
0,Lower finite edge,-0.06
1,Upper finite edge,0.06
2,Interior bin width,0.002
3,Interior bins,60
4,Tail bins,2
5,Total bins,62


,Input representation,Mean multiplier,Standard deviation multiplier,Edge multiplier,Maximum probability difference
0,Decimal returns,1,1,1,0
1,Percentage point returns,100,100,100,1.11022e-16


,Input representation,Mean multiplier,Standard deviation multiplier,Edge multiplier,Maximum probability difference
0,Decimal returns,1,1,1,0
1,Percentage point returns,100,100,100,1.11022e-16


In [8]:
figure, axis = plt.subplots(figsize=(8.5, 7.0), constrained_layout=True)
image = axis.imshow(pair_matrix, cmap="viridis", interpolation="nearest")
axis.set_xticks(np.arange(12), worked["Month name"], rotation=45, ha="right")
axis.set_yticks(np.arange(12), worked["Month name"])
axis.set_xlabel("Second monthly law")
axis.set_ylabel("First monthly law")
axis.set_title("Pair Contributions to Probability Ambiguity")
colorbar = figure.colorbar(image, ax=axis)
colorbar.set_label("Ordered pair contribution")
save_figure(figure, "pairwise_dispersion_1994")
display(figure)
plt.close(figure)

<Figure size 850x700 with 2 Axes>